# Setup

In [ ]:
import time
# Install the YAML magic
!pip install yamlmagic
%load_ext yamlmagic

In [ ]:
# Install the Kubeflow SDK (this can be removed once the latest version is included into workbench images)
!pip install git+https://github.com/kubeflow/trainer.git@release-1.9#subdirectory=sdk/python

# Training Configuration
#
Edit the following training parameters:

In [91]:
%%yaml parameters

# Model
model_name_or_path: microsoft/phi-2
model_revision: main
model_revision: main
torch_dtype: bfloat16
attn_implementation: eager    # one of eager (default), sdpa or flash_attention_2
use_liger: false                          # use Liger kernels

# PEFT / LoRA (Apply to Generator Model)
use_peft: true
lora_r: 16
lora_alpha: 8
lora_dropout: 0.05
lora_target_modules: ["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"] # Ensure these match your generator model
lora_modules_to_save: []

# QLoRA (BitsAndBytes) (Apply to Generator Model)
load_in_4bit: false                       # use 4 bit precision for the base model (only with LoRA)
load_in_8bit: false                       # use 8 bit precision for the base model (only with LoRA)

# Dataset
dataset_name: gsm8k                       # id or path to the dataset
dataset_config: main                      # name of the dataset configuration
dataset_train_split: train                # dataset split to use for training (for RAG generated data)
dataset_test_split: test                  # dataset split to use for evaluation (for RAG generated data)
dataset_kwargs:
  add_special_tokens: false               # template with special tokens
  append_concat_token: false              # add additional separator token

# SFT (These parameters will now apply to the RagModel's training)
max_seq_length: 1024                      # max sequence length for model and packing of the dataset
dataset_batch_size: 1000                  # samples to tokenize per batch (for initial data processing)
packing: false                            # Packing is generally not used directly with RagModel training in the same way as SFT

# Training
num_train_epochs: 3                       # number of training epochs (start small for RAG)
remove_unused_columns: false

per_device_train_batch_size: 1            # Batch size per device during training (RagModel can be memory intensive)
per_device_eval_batch_size: 1             # Batch size for evaluation
auto_find_batch_size: false               # find a batch size that fits into memory automatically
eval_strategy: epoch                      # evaluate every epoch

bf16: true                                # use bf16 16-bit (mixed) precision
tf32: false                               # use tf32 precision

learning_rate: 2.0e-5                     # Initial learning rate for RAG (often lower than SFT)
warmup_steps: 10                          # steps for a linear warmup from 0 to `learning_rate`
lr_scheduler_type: inverse_sqrt           # learning rate scheduler (see transformers.SchedulerType)

optim: adamw_torch_fused                  # optimizer (see transformers.OptimizerNames)
max_grad_norm: 1.0                        # max gradient norm
seed: 42

gradient_accumulation_steps: 4            # Increase for smaller per_device_train_batch_size
gradient_checkpointing: false             # use gradient checkpointing to save memory
gradient_checkpointing_kwargs:
  use_reentrant: false

# FSDP
fsdp: "full_shard auto_wrap"              # add offload if not enough GPU memory
fsdp_config:
  activation_checkpointing: true
  cpu_ram_efficient_loading: false
  sync_module_states: true
  use_orig_params: true
  limit_all_gathers: false

# Checkpointing
save_strategy: epoch                      # save checkpoint every epoch
save_total_limit: 1                       # limit the total amount of checkpoints
resume_from_checkpoint: false             # load the last checkpoint in output_dir and resume from it

# Logging
log_level: warning                        # logging level (see transformers.logging)
logging_strategy: steps
logging_steps: 1                          # log every N steps
report_to:
- tensorboard                             # report metrics to tensorboard

output_dir: /mnt/shared/fine_tuned_rag_model

<IPython.core.display.Javascript object>

# Feast Setup

In [ ]:
%%bash
pip install --quiet feast[milvus] sentence-transformers datasets faiss-cpu
pip install bigtree==0.19.2
pip install marshmallow==3.10.0
pip install git+https://github.com/feast-dev/feast.git@master

In [ ]:
from datasets import load_dataset
# load wikipedia dataset - 1% of the training split
dataset = load_dataset(
    "facebook/wiki_dpr",
    "psgs_w100.nq.exact",
    split="train[:1%]",
    with_index=False,
    trust_remote_code=True
)

In [63]:
def chunk_dataset(examples, chunk_size=100, overlap=20, max_chars=100):
    all_chunks = []
    all_ids = []
    all_titles = []

    for i, text in enumerate(examples['text']):  # Iterate over texts in the batch
        words = text.split()
        chunks = []
        for j in range(0, len(words), chunk_size - overlap):
            chunk_words = words[j:j + chunk_size]
            if len(chunk_words) < 20:
                continue
            chunk_text_value = ' '.join(chunk_words)  # Store the chunk text
            if len(chunk_text_value) > max_chars:
                chunk_text_value = chunk_text_value[:max_chars]
            chunks.append(chunk_text_value)
            all_ids.append(f"{examples['id'][i]}_{j}")  # Unique ID for the chunk
            all_titles.append(examples['title'][i])

        all_chunks.extend(chunks)

    return {'id': all_ids, 'title': all_titles, 'text': all_chunks}


chunked_dataset = dataset.map(
    chunk_dataset,
    batched=True,
    remove_columns=dataset.column_names,
    num_proc=1
)

Map:   0%|          | 0/210153 [00:00<?, ? examples/s]

In [64]:
from sentence_transformers import SentenceTransformer

sentences = chunked_dataset["text"]
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedding_model.encode(sentences, show_progress_bar=True, batch_size=64, device="cuda")

print(f"Generated embeddings of shape: {embeddings.shape}")

Batches:   0%|          | 0/6568 [00:00<?, ?it/s]

Generated embeddings of shape: (420306, 384)


In [67]:
import pandas as pd
from datetime import datetime, timezone

# Create DataFrame
df = pd.DataFrame({
    "passage_id": list(range(len(sentences))),
    "passage_text": sentences,
    "embedding": pd.Series(
        [embedding.tolist() for embedding in embeddings],
        dtype=object
    ),
    "event_timestamp": [datetime.now(timezone.utc) for _ in sentences],
})

print("DataFrame Info:")
print(df.head())
print(df["embedding"].apply(lambda x: len(x) if isinstance(x, list) else str(type(x))).value_counts())  # Check lengths

# Save to Parquet
df.to_parquet("feast_setup/data/wiki_dpr.parquet", index=False)
print("Saved to wiki_dpr.parquet")

DataFrame Info:
   passage_id                                       passage_text  \
0           0  Aaron Aaron ( or ; "Ahärôn") is a prophet, hig...   
1           1  Israelites, Aaron served as his brother's spok...   
2           2  God at Sinai granted Aaron the priesthood for ...   
3           3  could not speak well, God appointed Aaron as M...   
4           4  his rod turn into a snake. Then he stretched o...   

                                           embedding  \
0  [-0.03111468441784382, 0.08708751201629639, -0...   
1  [-0.03124287910759449, 0.13209620118141174, -0...   
2  [0.011147022247314453, 0.09298727661371231, -0...   
3  [0.03174533694982529, 0.12328037619590759, -0....   
4  [-0.02519603818655014, 0.07365845143795013, -0...   

                   event_timestamp  
0 2025-05-28 12:00:19.456182+00:00  
1 2025-05-28 12:00:19.456192+00:00  
2 2025-05-28 12:00:19.456192+00:00  
3 2025-05-28 12:00:19.456193+00:00  
4 2025-05-28 12:00:19.456193+00:00  
embedding
384   

In [68]:
%cd feast_setup

/opt/app-root/src/distributed-workloads/examples/kfto_feast_rag/feast_setup


In [66]:
%cd ..

/opt/app-root/src/distributed-workloads/examples/kfto_feast_rag


In [69]:
!feast apply

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


/opt/app-root/lib64/python3.11/site-packages/pymilvus/client/__init__.py:6: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import DistributionNotFound, get_distribution
/opt/app-root/lib64/python3.11/site-packages/pkg_resources/__init__.py:3147: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packages
  declare_namespace(pkg)
/opt/app-root/lib64/python3.11/site-packages/marshmallow/__init__.py:17: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  __version_info__ = tuple(LooseVersion(__version__).version)
/opt/app-root/lib64/python3.11/site-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "vector_e

In [ ]:
from feast import FeatureStore
import pandas as pd

store = FeatureStore(repo_path="")
df = pd.read_parquet("./data/wiki_dpr.parquet")
chunk_size = 10000
num_rows = len(df)

for i in range(0, num_rows, chunk_size):
    chunk_df = df.iloc[i:i + chunk_size]
    print(f"Writing chunk {i//chunk_size + 1}/{(num_rows + chunk_size - 1)//chunk_size}...")
    store.write_to_online_store(feature_view_name='wiki_passages', df=chunk_df)
    print("Chunk written successfully.")

print("All data written to online store.")

# Training Loop
#
Review the training function. You can adjust the chat template if needed depending on the model you want to fine-tune:

In [87]:
def main(parameters):
    import subprocess, sys
    # --- Install necessary packages ---
    # This ensures Feast and other dependencies are available inside the container
    print("Installing Feast and other RAG dependencies...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "feast[milvus]", "sentence-transformers", "datasets", "bigtree==0.19.2", "marshmallow==3.10.0", "protobuf>=3.10.0", "git+https://github.com/feast-dev/feast.git@master", "faiss-cpu"])
    print("Feast and other RAG dependencies installed.")
    from pathlib import Path
    import os
    import random
    from datasets import Dataset, load_dataset
    from transformers import (
        AutoTokenizer,
        set_seed,
        AutoModelForCausalLM,
        RagConfig,
        RagSequenceForGeneration,
        TrainingArguments,
        Trainer,
        AutoModel,
    )
    from peft import get_peft_model, prepare_model_for_kbit_training
    import numpy as np
    import torch

    from trl import (
        ModelConfig,
        ScriptArguments,
        SFTConfig,
        TrlParser,
        get_peft_config,
        get_quantization_config,
        get_kbit_device_map,
    )

    # This is required for `feast_rag_retriever` to be found (This is temporary until Feast RAG Retriever is part of Feast SDK)
    CUSTOM_MODULES_PATH = "/mnt/shared/kfto_feast_rag"
    sys.path.append(CUSTOM_MODULES_PATH)
    print(f"Added {CUSTOM_MODULES_PATH} to sys.path for custom module imports.")
    from feast_rag_retriever import FeastVectorStore, FeastRAGRetriever, FeastIndex
    from feast_setup.ragproject_repo import wiki_passage_feature_view

    parser = TrlParser((ScriptArguments, SFTConfig, ModelConfig))
    script_args, training_args_trl, model_args = parser.parse_dict(parameters)

    # Convert SFTConfig parameters to standard TrainingArguments
    training_args = TrainingArguments(
        output_dir=training_args_trl.output_dir,
        num_train_epochs=training_args_trl.num_train_epochs,
        per_device_train_batch_size=training_args_trl.per_device_train_batch_size,
        per_device_eval_batch_size=training_args_trl.per_device_eval_batch_size,
        gradient_accumulation_steps=training_args_trl.gradient_accumulation_steps,
        gradient_checkpointing=training_args_trl.gradient_checkpointing,
        learning_rate=training_args_trl.learning_rate,
        warmup_steps=training_args_trl.warmup_steps,
        lr_scheduler_type=training_args_trl.lr_scheduler_type,
        optim=training_args_trl.optim,
        max_grad_norm=training_args_trl.max_grad_norm,
        seed=training_args_trl.seed,
        bf16=training_args_trl.bf16,
        tf32=training_args_trl.tf32,
        eval_strategy=training_args_trl.eval_strategy,
        save_strategy=training_args_trl.save_strategy,
        save_total_limit=training_args_trl.save_total_limit,
        logging_strategy=training_args_trl.logging_strategy,
        logging_steps=training_args_trl.logging_steps,
        report_to=training_args_trl.report_to,
        fsdp=training_args_trl.fsdp if hasattr(training_args_trl, 'fsdp') else None,
        fsdp_config=training_args_trl.fsdp_config if hasattr(training_args_trl, 'fsdp_config') else None,
        resume_from_checkpoint=training_args_trl.resume_from_checkpoint,
        remove_unused_columns=training_args_trl.remove_unused_columns,
    )

    set_seed(training_args.seed)

    # --- Load Models for RAG ---
    # Question Encoder
    question_encoder_model_name_or_path = "sentence-transformers/all-MiniLM-L6-v2"
    question_encoder_tokenizer = AutoTokenizer.from_pretrained(question_encoder_model_name_or_path, trust_remote_code=model_args.trust_remote_code)
    question_encoder_model = AutoModel.from_pretrained(question_encoder_model_name_or_path, trust_remote_code=model_args.trust_remote_code)

    # Generator Model (The LLM to be fine-tuned)
    quantization_config = get_quantization_config(model_args)
    generator_tokenizer = AutoTokenizer.from_pretrained(
        parameters["model_name_or_path"], trust_remote_code=model_args.trust_remote_code, use_fast=True
    )
    if generator_tokenizer.pad_token is None:
        generator_tokenizer.pad_token = generator_tokenizer.eos_token

    generator_model = AutoModelForCausalLM.from_pretrained(
        parameters["model_name_or_path"],
        revision=model_args.model_revision,
        trust_remote_code=model_args.trust_remote_code,
        attn_implementation=model_args.attn_implementation,
        torch_dtype=model_args.torch_dtype,
        use_cache=False if training_args.gradient_checkpointing else True,
        device_map=get_kbit_device_map() if quantization_config is not None else None,
        quantization_config=quantization_config,
    )
    print(f"Generator Model Vocabulary Size: {generator_model.config.vocab_size}")

    if model_args.use_peft:
        generator_model = prepare_model_for_kbit_training(generator_model)
        peft_config = get_peft_config(model_args)
        generator_model = get_peft_model(generator_model, peft_config)
        print("PEFT setup for generator model completed.")

    store_path = CUSTOM_MODULES_PATH+"/feast_setup"

    # --- Initialize FeastRAGRetriever ---
    feast_index = FeastIndex()
    rag_top_k = 5
    question_encoder_config = {
        "model_type": "bert",
        "hidden_size": 384,
        "vocab_size": generator_tokenizer.vocab_size,
        "num_hidden_layers": 6,
        "num_attention_heads": 12
    }

    rag_config = RagConfig(
        question_encoder=question_encoder_config,
        generator=generator_model.config.to_dict(),
        index={"index_name": "feast_dummy_index", "custom_type": "FeastIndex"}
    )

    rag_retriever = FeastRAGRetriever(
        question_encoder_tokenizer=question_encoder_tokenizer,
        question_encoder=question_encoder_model,
        generator_tokenizer=generator_tokenizer,
        generator_model=generator_model,
        feast_repo_path=store_path,
        search_type="vector",
        id_field="passage_id",
        config=rag_config,
        index=feast_index,
        query_encoder_model="all-MiniLM-L6-v2",
    )

    # --- Initialize the RagModel for fine-tuning ---
    model = RagSequenceForGeneration(
        config=rag_config,
        generator=generator_model,
        retriever=rag_retriever
    )
    print("RAG Model initialized.")

    # Explicitly move model to GPU
    model = model.to('cuda')

    # --- Dataset Generation for Fine-tuning ---
    print("Preparing training data (Q&A pairs) with caching...")
    # Cache directory for synthetic data generated in a previous run to save time
    synthetic_data_cache_dir = "/mnt/shared/synthetic_data_cache/rag_synthetic_qa_dataset"
    Path(synthetic_data_cache_dir).mkdir(parents=True, exist_ok=True)
    cached_dataset_path = os.path.join(synthetic_data_cache_dir, "rag_synthetic_qa_dataset")

    if Path(cached_dataset_path).exists():
        print(f"Loading synthetic data from cache: {cached_dataset_path}")
        synthetic_dataset = Dataset.load_from_disk(cached_dataset_path)
    else:
        print("Generating training data for RAG fine-tuning (Q&A pairs)...")
        synthetic_data = []
        example_queries = [
            "What did Socrates say in his trial?",
            "When did the American Civil War begin?",
            "Who was the first person to walk on the moon?",
            "Explain the concept of quantum entanglement.",
            "What is the capital of France?",
            "How does photosynthesis work?",
            "Tell me about the history of the internet.",
            "What are the main causes of climate change?",
            "Who invented the telephone?",
            "What is the theory of relativity?",
            "Describe the life cycle of a butterfly.",
            "What is the function of the human heart?",
            "Explain the Big Bang theory.",
            "Who wrote 'Romeo and Juliet'?",
            "What is the significance of the Magna Carta?",
        ]

        MAX_TOKENIZER_LENGTH = 512

        for i, query in enumerate(example_queries):
            print(f"Generating answer for query {i+1}/{len(example_queries)}: '{query}'")
            try:
                rag_answer = rag_retriever.generate_answer(query, top_k=rag_top_k)
                full_text = f"Question: {query}\nAnswer: {rag_answer}"

                # Tokenize the full text
                tokenized_full_text = generator_tokenizer(
                    full_text,
                    truncation=True,
                    max_length=MAX_TOKENIZER_LENGTH,
                    return_tensors="np",
                )

                # Tokenize only the prompt part
                tokenized_prompt = generator_tokenizer(
                    f"Question: {query}\nAnswer:",
                    truncation=True,
                    max_length=MAX_TOKENIZER_LENGTH,
                    return_tensors="np",
                )

                # input_ids for model's embedding layer
                input_ids_for_model = tokenized_full_text["input_ids"].copy()
                # labels to compare and derive loss function
                labels_for_loss = tokenized_full_text["input_ids"].copy()

                # Apply the masking to labels_for_loss.
                if tokenized_prompt["input_ids"].shape[1] <= labels_for_loss.shape[1]:
                    labels_for_loss[0, :tokenized_prompt["input_ids"].shape[1]] = -100
                else:
                    print(f"Prompt length ({tokenized_prompt['input_ids'].shape[1]}) "
                          f"exceeds max_length for labels ({labels_for_loss.shape[1]}). "
                          "Masking entire sequence for labels.")
                    labels_for_loss[0, :] = -100

                synthetic_data.append({
                    "input_ids": input_ids_for_model.astype(np.int64).flatten().tolist(),
                    "attention_mask": tokenized_full_text["attention_mask"].astype(np.int64).flatten().tolist(),
                    "labels": labels_for_loss.astype(np.int64).flatten().tolist(),
                })

            except Exception as e:
                print(f"Error generating answer for query '{query}': {e}. Skipping this query.")

        synthetic_dataset = Dataset.from_list(synthetic_data)
        print(f"Synthetic data generation complete. Saving to cache: {cached_dataset_path}")
        synthetic_dataset.save_to_disk(cached_dataset_path)
        print("Synthetic data saved to cache.")

    train_dataset = synthetic_dataset.train_test_split(test_size=0.1, seed=training_args.seed)[script_args.dataset_train_split]
    test_dataset = synthetic_dataset.train_test_split(test_size=0.1, seed=training_args.seed)[script_args.dataset_test_split]

    with training_args.main_process_first(desc="Log few samples from the training set"):
        for index in random.sample(range(len(train_dataset)), 2):
            print(f"Sample Input IDs: {train_dataset[index]['input_ids'][:50]}...")
            print(f"Sample Labels: {train_dataset[index]['labels'][:50]}...")

    # --- Training the RagModel ---
    from transformers import DataCollatorWithPadding

    data_collator = DataCollatorWithPadding(tokenizer=generator_tokenizer, padding="longest", return_tensors="pt")

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        tokenizer=generator_tokenizer,
        data_collator=data_collator,
    )

    if trainer.accelerator.is_main_process:
        if hasattr(trainer.model.generator, "print_trainable_parameters"):
            print("Trainable parameters for PEFT-enabled generator:")
            trainer.model.generator.print_trainable_parameters()
        else:
            print("Trainer model does not have 'print_trainable_parameters' method on its generator.")

    checkpoint = None
    if training_args.resume_from_checkpoint is not None:
        checkpoint = training_args.resume_from_checkpoint

    print("Sample validation:")
    sample = train_dataset[0]
    print("Input IDs:", sample["input_ids"])
    print("Decoded:", generator_tokenizer.decode(sample["input_ids"]))

    try:
        # Test forward pass
        test_input = {k: torch.tensor(v).unsqueeze(0).cuda() for k,v in sample.items()}
        print("Testing forward pass input type:", test_input)
        with torch.no_grad():
            outputs = model(**test_input)
        print("Test forward pass successful")
    except Exception as e:
        print(f"Forward pass failed: {e}")
        raise

    print("Starting RAG model training...")
    trainer.train(resume_from_checkpoint=checkpoint)
    print("RAG model training completed.")

    trainer.save_model(training_args.output_dir)

    with training_args.main_process_first(desc="Training completed"):
        print(f"Training completed, model checkpoint written to {training_args.output_dir}")

# Training Client
#
Configure the SDK client by providing the authentication token:

In [ ]:
from kubernetes import client
from kubeflow.training import TrainingClient
from kubeflow.training.models import V1Volume, V1VolumeMount, V1PersistentVolumeClaimVolumeSource

api_server = "https://api.oai-kft-ibm.ibm.rh-ods.com:6443"
token = ""

configuration = client.Configuration()
configuration.host = api_server
configuration.api_key = {"authorization": f"Bearer {token}"}
# Un-comment if your cluster API server uses a self-signed certificate or an un-trusted CA
configuration.verify_ssl = False
api_client = client.ApiClient(configuration)
client = TrainingClient(client_configuration=api_client.configuration)

# Training Job
#
You're now almost ready to create the training job:
* Fill the `HF_HOME` environment variable with your HuggingFace token if you fine-tune a gated model
* Check the number of worker nodes
* Amend the resources per worker according to the job requirements
* If you use AMD accelerators:
  * Change `nvidia.com/gpu` to `amd.com/gpu` in `resources_per_worker`
  * Change `base_image` to `quay.io/modh/training:py311-rocm62-torch251`
* Update the PVC name to the one you've attached to the workbench if needed

In [89]:
client.delete_job(name="sft-rag-new")

/opt/app-root/lib64/python3.11/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.oai-kft-ibm.ibm.rh-ods.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


In [ ]:
client.create_job(
    job_kind="PyTorchJob",
    name="sft-rag-new",
    train_func=main,
    num_workers=1,
    num_procs_per_worker="1",
    resources_per_worker={
        "nvidia.com/gpu": 1,
        "memory": "64Gi",
        "cpu": 8,
    },
    base_image="quay.io/modh/training:py311-cuda124-torch251",
    env_vars={
        "CUDA_LAUNCH_BLOCKING": "1",
        "HF_HOME": "/mnt/shared/.cache",
        "HF_TOKEN": "",
        "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
        "PYTORCH_HIP_ALLOC_CONF": "expandable_segments:True",
        "NCCL_DEBUG": "INFO",
    },
    parameters=parameters,
    volumes=[
        V1Volume(name="shared",
                 persistent_volume_claim=V1PersistentVolumeClaimVolumeSource(claim_name="shared")),
    ],
    volume_mounts=[
        V1VolumeMount(name="shared", mount_path="/mnt/shared"),
    ],
)

Once the training job is created, you can follow its progress:

In [90]:
client.get_job_logs(
    name="sft-rag-new",
    job_kind="PyTorchJob",
    follow=True,
)

[Pod sft-rag-new-master-0]: bash: line 3: esa_rag: command not found
[Pod sft-rag-new-master-0]: Installing Feast and other RAG dependencies...
[Pod sft-rag-new-master-0]: [notice] A new release of pip is available: 24.2 -> 25.1.1
[Pod sft-rag-new-master-0]: [notice] To update, run: pip install --upgrade pip
[Pod sft-rag-new-master-0]: Feast and other RAG dependencies installed.
[Pod sft-rag-new-master-0]: [2025-05-28 14:55:07,998] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to cuda (auto detect)
[Pod sft-rag-new-master-0]: df: /opt/app-root/src/.triton/autotune: No such file or directory
[Pod sft-rag-new-master-0]: Added /mnt/shared/kfto_feast_rag to sys.path for custom module imports.
[Pod sft-rag-new-master-0]: [W528 14:55:10.663630085 CUDAAllocatorConfig.h:28] Warning: expandable_segments not supported on this platform (function operator())
[Pod sft-rag-new-master-0]: /opt/app-root/lib64/python3.11/site-packages/trl/trainer/sft_config.py:210: Deprecation

/opt/app-root/lib64/python3.11/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.oai-kft-ibm.ibm.rh-ods.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


({'sft-rag-new-master-0': 'bash: line 3: esa_rag: command not foundInstalling Feast and other RAG dependencies...[notice] A new release of pip is available: 24.2 -> 25.1.1[notice] To update, run: pip install --upgrade pipFeast and other RAG dependencies installed.[2025-05-28 14:55:07,998] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to cuda (auto detect)df: /opt/app-root/src/.triton/autotune: No such file or directoryAdded /mnt/shared/kfto_feast_rag to sys.path for custom module imports.[W528 14:55:10.663630085 CUDAAllocatorConfig.h:28] Warning: expandable_segments not supported on this platform (function operator())/opt/app-root/lib64/python3.11/site-packages/trl/trainer/sft_config.py:210: DeprecationWarning: `dataset_batch_size` is deprecated and will be removed in version 0.18.0. You can safely remove this parameter from your configuration.  warnings.warn(/opt/app-root/lib64/python3.11/site-packages/trl/trainer/sft_config.py:232: DeprecationWarning: `max_s

# TensorBoard
#
You can track your job runs and visualize the training metrics with TensorBoard:

In [ ]:
import os
os.environ["TENSORBOARD_PROXY_URL"]= os.environ["NB_PREFIX"]+"/proxy/6006/"

In [ ]:
%load_ext tensorboard

In [ ]:
%tensorboard --logdir /opt/app-root/src/shared

# Testing

## Testing the Fine-Tuned RAG Model
#
After fine-tuning, you can load the entire RAG model and test its performance by providing a question.
The `RagModel` will internally perform retrieval and then generate an answer based on the retrieved context.

In [ ]:
# Install / upgrade dependencies
!pip install --upgrade transformers peft accelerate sentence-transformers

In [ ]:
import torch
from transformers import AutoTokenizer, RagSequenceForGeneration, RagConfig, AutoModel, AutoModelForCausalLM
from peft import PeftModel
from feast import FeatureStore
from sentence_transformers import SentenceTransformer
from feast_rag_retriever import FeastVectorStore, FeastRAGRetriever, FeastIndex
from feast_setup.ragproject_repo import wiki_passage_feature_view


# This is required for `feast_rag_retriever` to be found (This is temporary until Feast RAG Retriever is part of Feast SDK)
CUSTOM_MODULES_PATH = "/mnt/shared/kfto_feast_rag"

store_path = CUSTOM_MODULES_PATH+"/feast_setup"
feast_index_inference = FeastIndex()
embedding_model_for_retriever_inference = SentenceTransformer("all-MiniLM-L6-v2") #
vector_store_inference = FeastVectorStore(store_repo_path=store_path, rag_view=wiki_passage_feature_view)


# Load the fine-tuned RAG model
finetuned_rag_model_path = "/mnt/shared/fine_tuned_rag_model/checkpoint"

print(f"Loading fine-tuned RAG model from: {finetuned_rag_model_path}")

rag_config_inference = RagConfig.from_pretrained(finetuned_rag_model_path)

generator_tokenizer_inference = AutoTokenizer.from_pretrained(finetuned_rag_model_path)
if generator_tokenizer_inference.pad_token is None:
    generator_tokenizer_inference.pad_token = generator_tokenizer_inference.eos_token

question_encoder_inference = AutoModel.from_pretrained(f"{finetuned_rag_model_path}/question_encoder")
generator_model_inference = AutoModelForCausalLM.from_pretrained(f"{finetuned_rag_model_path}/generator")

# Re-initialize the FeastRAGRetriever for inference
rag_retriever_inference = FeastRAGRetriever(
    question_encoder_tokenizer=AutoTokenizer.from_pretrained(rag_config_inference.question_encoder.name_or_path), # Use the actual question encoder tokenizer
    question_encoder=question_encoder_inference,
    generator_tokenizer=generator_tokenizer_inference,
    generator_model=generator_model_inference,
    feast_repo_path=store_path,
    vector_store=vector_store_inference,
    search_type="vector",
    config=rag_config_inference,
    index=feast_index_inference,
    query_encoder_model=embedding_model_for_retriever_inference
)

rag_retriever_inference = FeastRAGRetriever(
    question_encoder_tokenizer=AutoTokenizer.from_pretrained(rag_config_inference.question_encoder.name_or_path),
    question_encoder=question_encoder_inference,
    generator_tokenizer=generator_tokenizer_inference,
    generator_model=generator_model_inference,
    feast_repo_path=store_path,
    search_type="vector",
    id_field="passage_id",
    config=rag_config_inference,
    index=feast_index_inference,
    query_encoder_model="all-MiniLM-L6-v2",
)

finetuned_rag_model = RagSequenceForGeneration(
    config=rag_config_inference,
    question_encoder=question_encoder_inference,
    generator=generator_model_inference,
    retriever=rag_retriever_inference
) #

finetuned_rag_model.to("cuda" if torch.cuda.is_available() else "cpu")
finetuned_rag_model.eval() # Set model to evaluation mode

print("Fine-tuned RAG model loaded and ready for inference.")

# Test queries
test_query_1 = "What did Socrates say in his trial?"
test_query_2 = "Tell me about the Battle of Gettysburg."
test_query_3 = "When was the Declaration of Independence signed?"

print(f"\nQuery: {test_query_1}")
inputs = generator_tokenizer_inference(test_query_1, return_tensors="pt").input_ids.to(finetuned_rag_model.device)
generated_ids = finetuned_rag_model.generate(input_ids=inputs, num_beams=5, max_new_tokens=200)
answer_1 = generator_tokenizer_inference.decode(generated_ids[0], skip_special_tokens=True)
print(f"Answer: {answer_1}")

print(f"\nQuery: {test_query_2}")
inputs = generator_tokenizer_inference(test_query_2, return_tensors="pt").input_ids.to(finetuned_rag_model.device)
generated_ids = finetuned_rag_model.generate(input_ids=inputs, num_beams=5, max_new_tokens=200)
answer_2 = generator_tokenizer_inference.decode(generated_ids[0], skip_special_tokens=True)
print(f"Answer: {answer_2}")

print(f"\nQuery: {test_query_3}")
inputs = generator_tokenizer_inference(test_query_3, return_tensors="pt").input_ids.to(finetuned_rag_model.device)
generated_ids = finetuned_rag_model.generate(input_ids=inputs, num_beams=5, max_new_tokens=200)
answer_3 = generator_tokenizer_inference.decode(generated_ids[0], skip_special_tokens=True)
print(f"Answer: {answer_3}")


# Cleaning Up

## Training Job
#
Once you're done or want to re-create the training job, you can delete the existing one:

In [ ]:
client.delete_job(name="sft-rag")

## GPU Memory
#
If you want to start over and test the pre-trained model again, you can free the GPU / accelerator memory with:

In [ ]:
# Unload the model from GPU memory
import gc

del finetuned_rag_model, question_encoder_inference, generator_model_inference, rag_retriever_inference
del store_inference, vector_store_inference, feast_index_inference, embedding_model_for_retriever_inference

gc.collect()
torch.cuda.empty_cache()